# GEOL230 — Case Study: Yellowstone National Park (Student Starter)

This starter notebook has **scaffolding** for Tables 1–3 and Figures 1–4. You will fill in the **TODO** sections.

**Deliverables** (saved to `outputs/`):
- **Table 1** (XLSX and DOCX) Site-level summary for pH, Electrical Conductivity, Organic Matter, Bulk Density, Total Carbon using only 2016 data (compute mean and standard deviation).
- **Figure 1** (PNG) Clustered bar chart for pH, EC, and BD. Bars clustered by parameter (not by site). Include standard deviation error bars.
- **Figure 2** (PNG) Clustered bar chart for OM and TC. Bars clustered by parameter. Include standard deviation error bars.
- **Table 2** (XLSX and DOCX) Yearly average soil temperature for each summit/location (from the daily dataset).
- **Figure 3** (PNG) Daily temperatures for plot E22 at all sites (choose a single year or a 12-month window so the figure remains readable; justify your choice).
- **Table 3** (XLSX and DOCX) For Bechler River, compute min/max/mean/standard deviation and relevant sample counts (including discharge count).
- **Figure 4** (PNG) A descriptive water chemistry figure using a new figure type (not a clustered bar chart or a time series identical to Figure 3). Include a brief caption explaining why the figure is appropriate.



In [6]:
# --- Load Packages, Set Output Directory, & Create Save Helpers ---

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams["figure.dpi"] = 120

OUTPUT_DIR = "outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

def save_df_excel(df, filename, sheet_name="Table", *, index=True, merge_cells=True):
    """Save a DataFrame as an Excel file (.xlsx) in outputs/."""
    if not filename.lower().endswith(".xlsx"):
        filename = filename + ".xlsx"
    path = os.path.join(OUTPUT_DIR, filename)
    with pd.ExcelWriter(path, engine="openpyxl") as writer:
        df.to_excel(writer, sheet_name=sheet_name, index=index, merge_cells=merge_cells)
    print(f"Saved: {path}")
    return path

def save_fig(filename):
    """Save the current matplotlib figure to outputs/."""
    path = os.path.join(OUTPUT_DIR, filename)
    plt.savefig(path, dpi=300, bbox_inches="tight")
    print(f"Saved: {path}")
    return path

def checkpoint(title, df, cols=None, n=5):
    """Quick, friendly printout to prove a step worked."""
    print(f"\n It worked - you're so smart! {title}")
    print("shape:", df.shape)
    if cols:
        for c in cols:
            if c in df.columns:
                try:
                    print(f"- {c} unique (first 10):", list(pd.Series(df[c].dropna().unique())[:10]))
                except Exception as e:
                    print(f"- {c}: (could not summarize) {e}")
    display(df.head(n))


In [ ]:
# --- Load the Excel workbook ---
# 1) Click the folder icon (left sidebar) to see files
# 2) Upload the Excel file to this Colab runtime
# 3) Update FILE_NAME below if needed

FILE_NAME = "LeMonte_Josh_GEOL230_YNP Lab.xlsx"

# If the file isn't present, try a Colab upload prompt
if not os.path.exists(FILE_NAME):
    from google.colab import files
    uploaded = files.upload()
    # If the exact name isn't present, use the first uploaded file
    if FILE_NAME not in uploaded:
        FILE_NAME = list(uploaded.keys())[0]
        print("Using uploaded file:", FILE_NAME)

# Peek at available sheet names so you can choose the right ones
xls = pd.ExcelFile(FILE_NAME)
print("Sheets found in workbook:")
for s in xls.sheet_names:
    print(" -", s)

# HINT: In this workbook, the cleaned sheets are usually:
#   - Clean Soil Chem (YNP)
#   - Clean Soil Temp (YNP)
#   - Clean Water Chem (YNP)

# TODO: set these to the correct sheet names (copy/paste from the list above)
SOIL_SHEET  = ""   # e.g., "Soil Chemistry"
TEMP_SHEET  = ""   # e.g., "Soil Temperature"
WATER_SHEET = ""   # e.g., "Water Chemistry"

# Load sheets
soil_df  = pd.read_excel(FILE_NAME, sheet_name=SOIL_SHEET)
temp_df  = pd.read_excel(FILE_NAME, sheet_name=TEMP_SHEET)
water_df = pd.read_excel(FILE_NAME, sheet_name=WATER_SHEET)

checkpoint("Loaded soil_df", soil_df)
checkpoint("Loaded temp_df", temp_df)
checkpoint("Loaded water_df", water_df)


## Quick peek at columns
Run these cells so you know what column names exist.


In [ ]:
print("Soil chem columns:", list(soil_df.columns))
print("\nSoil temp columns:", list(temp_df.columns))
print("\nWater chem columns (first 25):", list(water_df.columns)[:25])


## Mini pandas crash course (you only need 4 moves)

In this case study, you will repeatedly use **four pandas moves**:

1) **Filter rows**: keep only the rows you want  
2) **Keep certain categories** with `.isin(...)`  
3) **Rename/translate values** with `.map(...)`  
4) **Convert text to numbers** with `pd.to_numeric(..., errors="coerce")`

### Read this like a recipe
After each step, run a **checkpoint** so you can *prove* the step worked (shape + a few unique values).


### Practice on a tiny “toy” dataset first (no stakes)

Before you touch the real Yellowstone dataset, we’ll practice on a **mini dataset** that has the same *kinds* of columns and common problems:

- **Wrong year rows** you need to filter out
- **Non‑YNP sites** you need to exclude
- **Parameters you don’t need** (and a few you do)
- **Messy numbers** like `"n/a"` or `"bad"` that should become `NaN`

#### The only goal
By the end, you should be able to produce a 4‑column output:

`SiteName, Parameter, Mean, SD`

#### How to use this section
1. **Run the “demo” cell** once and watch the checkpoints.
2. Then do the **“your turn” cell** (you’ll fill in a few blanks).  
   If you get stuck, use the checkpoints to find *which step* broke.

> Tip: After each step, always ask: **Did rows change? Did columns change?**


In [8]:
# DEMO CELL
# --- Toy dataset: learn the 4 pandas moves (with checkpoints) ---
# Read this cell like a recipe. After each step:
#   1) run the line(s)
#   2) look at the checkpoint output
#   3) answer: "What changed (rows/columns/values)?"

toy = pd.DataFrame({
    "YearSampled": [2016, 2016, 2015, 2016, 2017, 2016, 2016, 2016],
    "SiteName": ["YNP-01", "YNP-02", "YNP-01", "ABC-99", "YNP-03", "YNP-04", None, "YNP-05"],
    "ParameterDataset": ["pH", "EC", "pH", "pH", "C", "Bulk_Density", "OrganicMatter", "EC"],
    "Average": ["7.1", "120", "6.8", "8.0", "1.9", "0.92", "n/a", "115"],
    "StDev": ["0.2", "5", "0.1", "0.3", "0.4", "0.05", "0.8", "bad"],
})

PARAM_MAP = {
    "pH": "pH",
    "EC": "EC",
    "OrganicMatter": "Organic Matter",
    "Bulk_Density": "Bulk Density",
    "C": "Total Carbon",
}

checkpoint("Toy data (starting point)", toy, cols=["YearSampled","SiteName","ParameterDataset"])

# STEP 1 — Filter rows by year
# Predict: should the number of rows go down?
toy2016 = toy[toy["YearSampled"] == 2016]
checkpoint("Toy: after YearSampled == 2016", toy2016, cols=["YearSampled"])

# STEP 2 — Filter rows by site name substring
# Why astype(str)? Because SiteName includes a None (missing value) and .str.contains would error otherwise.
toy2016 = toy2016[toy2016["SiteName"].astype(str).str.contains("YNP")]
checkpoint("Toy: after SiteName contains 'YNP'", toy2016, cols=["SiteName"])

# STEP 3 — Keep only certain categories (parameters) using isin
# Predict: will the set of ParameterDataset values shrink?
toy2016 = toy2016[toy2016["ParameterDataset"].isin(PARAM_MAP.keys())].copy()
checkpoint("Toy: after ParameterDataset isin(PARAM_MAP.keys())", toy2016, cols=["ParameterDataset"])

# STEP 4 — Translate coded values into friendly labels using map
toy2016["Parameter"] = toy2016["ParameterDataset"].map(PARAM_MAP)

# STEP 5 — Convert text to numbers safely
# errors="coerce" means: if conversion fails, pandas puts NaN instead of crashing.
toy2016["Mean"] = pd.to_numeric(toy2016["Average"], errors="coerce")
toy2016["SD"]   = pd.to_numeric(toy2016["StDev"],   errors="coerce")

checkpoint("Toy: after map + to_numeric", toy2016, cols=["Parameter","Mean","SD"])

# STEP 6 — Select only the columns we need for the final table
toy_final = toy2016[["SiteName","Parameter","Mean","SD"]]
display(toy_final)



 It worked - you're so smart! Toy data (starting point)
shape: (8, 5)
- YearSampled unique (first 10): [2016, 2015, 2017]
- SiteName unique (first 10): ['YNP-01', 'YNP-02', 'ABC-99', 'YNP-03', 'YNP-04', 'YNP-05']
- ParameterDataset unique (first 10): ['pH', 'EC', 'C', 'Bulk_Density', 'OrganicMatter']


,YearSampled,SiteName,ParameterDataset,Average,StDev
0,2016,YNP-01,pH,7.1,0.2
1,2016,YNP-02,EC,120,5
2,2015,YNP-01,pH,6.8,0.1
3,2016,ABC-99,pH,8.0,0.3
4,2017,YNP-03,C,1.9,0.4



 It worked - you're so smart! Toy: after YearSampled == 2016
shape: (6, 5)
- YearSampled unique (first 10): [2016]


,YearSampled,SiteName,ParameterDataset,Average,StDev
0,2016,YNP-01,pH,7.1,0.2
1,2016,YNP-02,EC,120,5
3,2016,ABC-99,pH,8.0,0.3
5,2016,YNP-04,Bulk_Density,0.92,0.05
6,2016,None,OrganicMatter,n/a,0.8



 It worked - you're so smart! Toy: after SiteName contains 'YNP'
shape: (4, 5)
- SiteName unique (first 10): ['YNP-01', 'YNP-02', 'YNP-04', 'YNP-05']


...


 It worked - you're so smart! Toy: after ParameterDataset isin(PARAM_MAP.keys())
shape: (4, 5)
- ParameterDataset unique (first 10): ['pH', 'EC', 'Bulk_Density']


,YearSampled,SiteName,ParameterDataset,Average,StDev
0,2016,YNP-01,pH,7.1,0.2
1,2016,YNP-02,EC,120,5
5,2016,YNP-04,Bulk_Density,0.92,0.05
7,2016,YNP-05,EC,115,bad



 It worked - you're so smart! Toy: after map + to_numeric
shape: (4, 8)
- Parameter unique (first 10): ['pH', 'EC', 'Bulk Density']
- Mean unique (first 10): [7.1, 120.0, 0.92, 115.0]
- SD unique (first 10): [0.2, 5.0, 0.05]


,YearSampled,SiteName,ParameterDataset,Average,StDev,Parameter,Mean,SD
0,2016,YNP-01,pH,7.1,0.2,pH,7.10,0.20
1,2016,YNP-02,EC,120,5,EC,120.00,5.00
5,2016,YNP-04,Bulk_Density,0.92,0.05,Bulk Density,0.92,0.05
7,2016,YNP-05,EC,115,bad,EC,115.00,NaN


,SiteName,Parameter,Mean,SD
0,YNP-01,pH,7.10,0.20
1,YNP-02,EC,120.00,5.00
5,YNP-04,Bulk Density,0.92,0.05
7,YNP-05,EC,115.00,NaN


### Your turn (on the toy data)

In the next cell, you will recreate the pipeline yourself.

Rules:
- **Do not** copy/paste from above line-for-line.
- After each step, run the checkpoint and confirm it matches what you expect.

If your result is wrong, the checkpoints tell you **where** it went wrong.


In [ ]:
# --- Your turn: recreate the pipeline on the toy dataset ---

toy2 = toy.copy()

# 1) Filter to 2016
# TODO: replace ____ with the correct value
toy2 = toy2[toy2["YearSampled"] == ____]
checkpoint("Your turn: after year filter", toy2, cols=["YearSampled"])

# 2) Keep only SiteName values that contain "YNP"
toy2 = toy2[toy2["SiteName"].astype(str).str.contains(____)]
checkpoint("Your turn: after site filter", toy2, cols=["SiteName"])

# 3) Keep only parameters listed in PARAM_MAP
toy2 = toy2[toy2["ParameterDataset"].isin(____)].copy()
checkpoint("Your turn: after parameter filter", toy2, cols=["ParameterDataset"])

# 4) Make readable names + numeric columns
toy2["Parameter"] = toy2["ParameterDataset"].map(____)
toy2["Mean"] = pd.to_numeric(toy2["Average"], errors=____)
toy2["SD"]   = pd.to_numeric(toy2["StDev"],   errors=____)

checkpoint("Your turn: after map + to_numeric", toy2, cols=["Parameter","Mean","SD"])

toy2_final = toy2[["SiteName","Parameter","Mean","SD"]]
display(toy2_final)

# --- Quick self-check (should run without errors when you filled blanks correctly) ---
assert set(toy2_final.columns) == {"SiteName","Parameter","Mean","SD"}
assert toy2_final["SiteName"].astype(str).str.contains("YNP").all()
assert (toy2_final["Mean"].dtype.kind in "fi") and (toy2_final["SD"].dtype.kind in "fi")


In [ ]:
# --- Table 1 Prep: Filter to 2016 + Yellowstone + needed parameters ---

# Copy the parameter map here so you can reuse it in later cells
PARAM_MAP = {
    "pH": "pH",
    "EC": "EC",
    "OrganicMatter": "Organic Matter",
    "Bulk_Density": "Bulk Density",
    "C": "Total Carbon",
}

# TODO: Create a filtered DataFrame called soil2016 with ONLY:
# - YearSampled == 2016
# - Yellowstone sites (SiteName contains "YNP")
# - Parameters in PARAM_MAP (use .isin)
#
# TIP: Do it as 3–4 short filters, and run checkpoint() after each one.

# soil2016 = soil_df.copy()
# soil2016 = ...
# checkpoint("After Year filter", soil2016, cols=["YearSampled"])
# soil2016 = ...
# checkpoint("After SiteName filter", soil2016, cols=["SiteName"])
# soil2016 = ...
# checkpoint("After Parameter filter", soil2016, cols=["ParameterDataset"])

# TODO: Add derived columns:
# - Parameter (mapped from ParameterDataset)
# - Mean (numeric from Average)
# - SD (numeric from StDev)
#
# soil2016["Parameter"] = ...
# soil2016["Mean"] = ...
# soil2016["SD"]   = ...
# checkpoint("After mapping + numeric conversion", soil2016, cols=["Parameter","Mean","SD"])

# Preview the final minimal columns
# soil2016[["SiteName","Parameter","Mean","SD"]].head()


## Table 1 + Figures 1–2 (Soil Chemistry, 2016 only)

**Requirement summary**
- **Table 1:** Site-level summary for **pH, EC, Organic Matter, Bulk Density, Total Carbon** using **2016 data** (mean and standard deviation).
- **Figure 1:** Clustered bar chart for **pH, EC, BD**; clustered by **parameter**; include **SD error bars**.
- **Figure 2:** Clustered bar chart for **OM, TC**; clustered by **parameter**; include **SD error bars**.

**Note about the Clean Soil Chem sheet:** it already includes `Average` and `StDev` columns for each site/parameter sampling event, so we treat:
- `Mean` = `Average`
- `SD`   = `StDev`

We also filter to Yellowstone sites (`US_YNP_*`) to match the YNP-focused deliverables.
### Your tasks
1. Filter soil data to **YearSampled == 2016**.
2. Filter to Yellowstone sites (`US_YNP_*`).
3. Keep only the needed parameters.
4. Build **Table 1** (mean + SD) and save as XLSX. You will use this to create a DOCX file later.
5. Make **Figure 1** and **Figure 2** (clustered by parameter) and save them.


In [ ]:
# TODO: Create a filtered DataFrame called soil2016 with ONLY:
# - YearSampled == 2016
# - Yellowstone sites (SiteName contains "YNP")
# - Parameters: pH, EC, OrganicMatter, Bulk_Density, C
#
# HINT: Use .copy() at the end to avoid SettingWithCopy warnings.

PARAM_MAP = {
    "pH": "pH",
    "EC": "EC",
    "OrganicMatter": "Organic Matter",
    "Bulk_Density": "Bulk Density",
    "C": "Total Carbon",
}

# --- TODO: filter ---
soil2016 = None  # <- replace with your filtered dataframe

# TODO: add these columns:
# soil2016["Parameter"] = ...
# soil2016["Mean"] = ...
# soil2016["SD"] = ...

# --- sanity check (should be 4 sites x 5 parameters = 20 rows, if your filters match Yellowstone) ---
# print(soil2016.shape)


In [ ]:
# TODO: Build Table 1 in wide format with Mean/SD for each parameter.
# Target structure: index=SiteName, columns are a MultiIndex with ("Mean"/"SD") and parameters.

# 1) pivot mean and sd tables
# t1_mean = ...
# t1_sd = ...

# 2) re-order parameters (pH, EC, Organic Matter, Bulk Density, Total Carbon)
col_order = ["pH","EC","Organic Matter","Bulk Density","Total Carbon"]

# 3) combine into one table called `table1`
table1 = None  # <- replace

# display(table1)

# TODO: save as .XLSX
# save_df_excel(table1, "Table1_SoilChem_2016_SiteSummary.xlsx")



In [ ]:
# TODO: Figure 1 — clustered bar chart (clustered by parameter) for pH, EC, Bulk Density.
# HINT: Use pivot so index=Parameter, columns=SiteName, values=Mean; do the same for SD.
# HINT: pandas .plot(kind="bar", yerr=..., capsize=3) works well.

# f1 = soil2016[soil2016["Parameter"].isin(["pH","EC","Bulk Density"])].copy()
# mean_w = ...
# sd_w = ...
# ax = mean_w.plot(kind="bar", yerr=sd_w, capsize=3)
# ax.set_xlabel("Parameter"); ax.set_ylabel("..."); ax.set_title(...); REMEMBER that the figure caption can serve as the figure title
# ax.legend(...)
# plt.tight_layout()
# save_fig("Figure1_SoilChem_pH_EC_BD_2016.png")
# plt.show()

# TODO: Figure 2 — clustered bar chart for Organic Matter and Total Carbon (mean ± SD).


## Table 2 + Figure 3 (Soil Temperature, daily dataset)

**Requirement summary**
- **Table 2:** Yearly average soil temperature for each summit/location (from the daily dataset).
- **Figure 3:** Daily temperatures for **plot E22** at all sites (choose **one year** or **one 12‑month window** so it stays readable; justify choice).

We filter to Yellowstone (`Park == "YELL"`) and compute:
- yearly mean of `DailyMean` by `Summit` and `Year`
- Figure 3 uses **2016** because it is a full year with complete daily coverage for E22 in this dataset and aligns with the soil chemistry year.
### Your tasks
1. Convert `DateTime` to datetime and create a `Year` column.
2. Filter to Yellowstone (`Park == 'YELL'`).
3. Build **Table 2** (yearly mean temp for each Summit/location).
4. Build **Figure 3** for plot **E22** at all sites for a single year.


In [ ]:
# TODO: Convert DateTime to datetime and add Year column
# temp_df["DateTime"] = ...
# temp_df["Year"] = ...

# TODO: filter to Yellowstone
# temp_yell = ...

# TODO: Table 2: groupby Summit and Year, compute mean DailyMean
table2 = None  # <- replace
# display(table2.head())
# save_df_excel(table2, "Table2_YearlyAvgSoilTemp_bySummit.xlsx")


In [ ]:
# TODO: Figure 3 — daily temperatures for Plot E22 at all sites.
# Pick ONE year (or one 12-month window). Keep it readable and write a one-sentence justification.

YEAR_FOR_PLOT = 2016  # you may change this, but justify your choice

# e22 = temp_yell[(temp_yell["Plot"]=="E22") & (temp_yell["Year"]==YEAR_FOR_PLOT)].copy()

# fig, ax = plt.subplots(figsize=(10,4))
# for summit, g in e22.groupby("Summit"):
#     g = g.sort_values("DateTime")
#     ax.plot(g["DateTime"], g["DailyMean"], label=summit, linewidth=1)

# ax.set_title(...)
# ax.set_xlabel("Date"); ax.set_ylabel("Daily Mean Temp (°C)")
# ax.legend(...)
# plt.tight_layout()
# save_fig(f"Figure3_DailyTemp_E22_{YEAR_FOR_PLOT}.png")
# plt.show()

# print("Justification: ...")


## Table 3 + Figure 4 (Bechler River water chemistry)

**Requirement summary**
- **Table 3:** For **Bechler River**, compute min/max/mean/standard deviation and relevant sample counts (**including discharge count**).
- **Figure 4:** A descriptive water chemistry figure using a **new figure type** (not a clustered bar chart or a time series identical to Fig. 3). Include a brief caption explaining why it’s appropriate.

We’ll:
1) Clean the sheet (drop the first “units” row, coerce `---` to NaN, convert numeric columns)
2) Filter to `Area == "Bechler River"`
3) Compute summary statistics for a “core set” of common water-chem variables
4) Make a **correlation heatmap** (new figure type)
### Your tasks
1. Remove the units row (the row where `ID` is missing).
2. Filter to **Area == 'Bechler River'**.
3. Convert the chemistry columns you use to numeric (treat `'---'` as missing).
4. Build **Table 3** and save.
5. Create **Figure 4** using a new figure type (suggestion: correlation heatmap).


In [ ]:
# TODO: Clean + filter to Bechler River
w = water_df.copy()

# 1) drop units row
# w = ...

# 2) parse date
# w["Collection date"] = ...

# 3) filter area
# bech = ...

CORE_VARS = [
    "Discharge","pH","Specific conductance","Temperature",
    "Calcium (Ca)","Magnesium (Mg)","Sodium (Na)","Potassium (K)",
    "Chloride (Cl)","Sulfate (SO4)","Alkalinity","Silica (SiO2)",
    "Fluoride (F)","Bromide (Br)","Nitrate (NO3)","Ammonium (NH4)",
    "Dissolved Organic Carbon (DOC)","Phosphorus (P)","Arsenic (As)"
]

# TODO: convert to numeric, treating '---' as NaN
# for c in CORE_VARS:
#     bech[c] = ...

# TODO: Table 3 stats
table3 = None  # <- replace
# display(table3)
# save_df_excel(table3, "Table3_BechlerRiver_SummaryStats.xlsx")
# save_df_as_image(table3, "Table3_BechlerRiver_SummaryStats.png", fontsize=7, max_rows=30)

# TODO: Figure 4 (new figure type suggestions: correlation heatmap, )
# corr = ...
# fig, ax = plt.subplots(...)
# im = ax.imshow(...)
# ...
# save_fig("Figure4_BechlerRiver_CorrelationHeatmap.png")
# plt.show()
#
# print("Caption: ... why this figure type is appropriate")
